In [0]:
# Databricks notebook source
# =============================================================================
# GOLD LAYER — gold_base_enriched_transactions
# CELL 1: Configuration & Schema Setup
# =============================================================================
# This notebook builds the shared transaction foundation for all downstream
# Gold marts. It enriches every transaction with customer, account, card,
# merchant, device, and calendar context — plus fraud helper columns.
# =============================================================================

from datetime import datetime
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# --- config ---
catalog = "ubuntu_bank_fraud"
silver = f"{catalog}.silver"
bronze = f"{catalog}.bronze"
gold = f"{catalog}.gold"
batch_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Gold layer build started: {batch_id}")
print(f"Source: {silver}")
print(f"Target: {gold}")

# make sure gold schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")
print("Gold schema ready.\n")

Gold layer build started: 20260623_143823
Source: ubuntu_bank_fraud.silver
Target: ubuntu_bank_fraud.gold
Gold schema ready.



In [0]:
# Databricks notebook source
# =============================================================================
# CELL 2: Load Source DataFrames
# =============================================================================
# Loads silver transaction tables, all dimension tables, and the date
# dimension from bronze. SCD Type-2 tables are filtered to current records
# only using is_current = 1 (INT, not BOOLEAN).
# =============================================================================

# --- transaction tables ---
txn_fact     = spark.table(f"{silver}.silver_transaction_fact")
txn_features = spark.table(f"{silver}.silver_transaction_features")

# --- dimensions with SCD Type-2 (is_current is an integer: 1 = current) ---
cust_current = spark.table(f"{silver}.silver_customers") \
    .filter(col("is_current") == 1)

acct_current = spark.table(f"{silver}.silver_accounts") \
    .filter(col("is_current") == 1)

dev_current = spark.table(f"{silver}.silver_devices") \
    .filter(col("is_current") == 1)

# --- dimensions without SCD ---
cards_df       = spark.table(f"{silver}.silver_cards")
merchants_df   = spark.table(f"{silver}.silver_merchants")
device_links   = spark.table(f"{silver}.silver_customer_device_link")

# --- date dimension (lives in bronze, not silver) ---
date_dim = spark.table(f"{bronze}.brz_date_dim") \
    .select(
        col("FULL_DATE").cast("date").alias("full_date"),
        col("DAY_OF_WEEK"),
        col("MONTH"),
        col("MONTH_NAME"),
        col("QUARTER"),
        col("YEAR"),
        col("IS_WEEKEND").cast("int"),
        col("IS_HOLIDAY_SEASON").cast("int"),
        col("IS_MONTH_END").cast("int")
    )

# --- print what we have ---
print(f"txn_fact:       {txn_fact.count():,}")
print(f"txn_features:   {txn_features.count():,}")
print(f"customers:      {cust_current.count():,}  (current only)")
print(f"accounts:       {acct_current.count():,}  (current only)")
print(f"cards:          {cards_df.count():,}")
print(f"merchants:      {merchants_df.count():,}")
print(f"devices:        {dev_current.count():,}  (current only)")
print(f"device_links:   {device_links.count():,}")
print(f"date_dim:       {date_dim.count():,}")
print("All sources loaded.\n")

txn_fact:       2,200,000
txn_features:   2,200,000
customers:      30,000  (current only)
accounts:       45,000  (current only)
cards:          39,705
merchants:      2,200
devices:        28,000  (current only)
device_links:   34,706
date_dim:       731
All sources loaded.



In [0]:
# Databricks notebook source
# =============================================================================
# CELL 3: Account-Customer Enrichment
# =============================================================================
# Joins accounts to customers so every transaction can carry full customer
# context. We keep only the columns that fraud analytics actually needs.
# =============================================================================

acct_cust = acct_current \
    .select(
        "account_id",
        "customer_id",
        col("account_type"),
        col("account_status"),
        col("opening_date").alias("account_opened"),
        col("current_balance").alias("acct_balance"),
        col("credit_limit").alias("acct_credit_limit")
    ) \
    .join(
        cust_current.select(
            "customer_id",
            col("first_name"),
            col("last_name"),
            concat_ws(" ", "first_name", "last_name").alias("cust_full_name"),
            col("customer_segment"),
            col("residential_province").alias("cust_province"),
            col("residential_city").alias("cust_city"),
            col("onboarding_date").alias("cust_onboarded"),
            col("kyc_risk_score").alias("cust_kyc_score"),
            col("employment_status").alias("cust_employment"),
            col("monthly_income_band").alias("cust_income_band")
        ),
        "customer_id",
        "inner"
    )

print(f"Account-customer mapping: {acct_cust.count():,} rows")
acct_cust.select("account_id", "customer_id", "cust_full_name", "customer_segment").show(5, truncate=False)

Account-customer mapping: 45,000 rows
+------------+------------+--------------+----------------+
|account_id  |customer_id |cust_full_name|customer_segment|
+------------+------------+--------------+----------------+
|ACCT00000001|CUST00000001|TSHEPO MOLEFE |Mass_Retail     |
|ACCT00000002|CUST00000002|MUSA MTHEMBU  |Mass_Retail     |
|ACCT00000003|CUST00000003|PHUMLA NDLOVU |Affluent        |
|ACCT00000004|CUST00000004|NTOMBI MASEKO |Mass_Retail     |
|ACCT00000005|CUST00000005|DINEO MOLEPO  |Premier         |
+------------+------------+--------------+----------------+
only showing top 5 rows


In [0]:
# Databricks notebook source
# =============================================================================
# CELL 4: Device Enrichment
# =============================================================================
# Maps customers to devices through the link table. A single device used by
# many unrelated customers is a strong fraud ring indicator — we preserve
# the link type and device risk attributes.
# =============================================================================

cust_device = device_links \
    .select(
        "customer_id",
        "device_id",
        col("link_type").alias("dev_link_type"),
        col("is_primary_device")
    ) \
    .join(
        dev_current.select(
            "device_id",
            col("device_type"),
            col("operating_system").alias("dev_os"),
            col("device_manufacturer").alias("dev_mfr"),
            col("device_model"),
            col("device_risk_score").alias("dev_risk_score"),
            col("is_emulator"),
            col("is_rooted"),
            col("first_seen_date").alias("dev_first_seen"),
            col("geolocation_province").alias("dev_province")
        ),
        "device_id",
        "inner"
    )

print(f"Customer-device mapping: {cust_device.count():,} rows")
cust_device.groupBy("device_type").count().orderBy(col("count").desc()).show()

Customer-device mapping: 34,706 rows
+---------------+-----+
|    device_type|count|
+---------------+-----+
| ANDROID_MOBILE|19169|
|  IPHONE_MOBILE| 8556|
| WINDOWS_LAPTOP| 2867|
| ANDROID_TABLET| 1694|
|    IPAD_TABLET| 1075|
| MACBOOK_LAPTOP|  655|
|WINDOWS_DESKTOP|  509|
|    MAC_DESKTOP|  181|
+---------------+-----+



In [0]:
# Databricks notebook source
# =============================================================================
# CELL 5: Card & Merchant Enrichment
# =============================================================================
# Lightweight enrichment — we pick only the columns that add investigative
# value. No point carrying every attribute into the Gold base table.
# =============================================================================

# --- cards: keep only what an investigator would look at ---
card_lookup = cards_df.select(
    "card_id",
    "card_type",
    "card_network",
    "card_status",
    col("is_contactless").alias("card_contactless")
)

# --- merchants: name, category, risk band, director (for linkage) ---
merchant_lookup = merchants_df.select(
    "merchant_id",
    col("merchant_name"),
    col("merchant_category_code").alias("mcc"),
    col("merchant_category_desc").alias("merchant_category"),
    col("business_province").alias("merch_province"),
    col("risk_band").alias("merch_risk_band"),
    col("director_name").alias("merch_director"),
    col("director_id_number").alias("merch_director_id")
)

# --- date dimension ---
date_lookup = date_dim.select(
    col("full_date").alias("date_key"),
    col("DAY_OF_WEEK").alias("dow"),
    col("MONTH").alias("cal_month"),
    col("MONTH_NAME").alias("cal_month_name"),
    col("QUARTER").alias("cal_quarter"),
    col("YEAR").alias("cal_year"),
    col("IS_WEEKEND").cast("boolean").alias("is_weekend"),
    col("IS_HOLIDAY_SEASON").cast("boolean").alias("is_holiday"),
    col("IS_MONTH_END").cast("boolean").alias("is_month_end")
)

print(f"Card lookup:     {card_lookup.count():,}")
print(f"Merchant lookup: {merchant_lookup.count():,}")
print(f"Date lookup:     {date_lookup.count():,}")
print("Enrichment tables ready.\n")

Card lookup:     39,705
Merchant lookup: 2,200
Date lookup:     731
Enrichment tables ready.



In [0]:
# Databricks notebook source
# =============================================================================
# CELL 6: Assemble the Enriched Transaction Base
# =============================================================================
# The problem: silver_transaction_fact and silver_transaction_features share
# many columns (source_account_id, dest_account_id, transaction_date, 
# transaction_amount, transaction_type, channel, etc). When we join them on
# transaction_id, Spark sees duplicate columns from both sides.
#
# The fix: select only the FEATURE-SPECIFIC columns from txn_features before
# joining — the velocity metrics, flags, and aggregated values that exist
# ONLY in the features table, not in the fact table.
#
# Note: .cache() is removed because serverless compute does not support
# DataFrame persistence. The notebook is structured so each cell that needs
# the base DataFrame will reference the same transformation chain and Spark
# will optimize it under the hood.
# =============================================================================

print("Building enriched transaction base...")

# --- figure out which columns are unique to features ---
fact_cols = set(txn_fact.columns)
feat_cols = set(txn_features.columns)

# columns that exist in BOTH tables (Spark will keep one copy from the join key)
shared_cols = fact_cols & feat_cols
print(f"Shared columns between fact and features: {len(shared_cols)}")

# columns unique to features — these are the ones we actually need from features
feature_only_cols = [c for c in txn_features.columns if c not in shared_cols]
feature_only_cols.append("transaction_id")  # need this for the join
print(f"Feature-only columns we keep: {len(feature_only_cols) - 1}")

# step 1: fact + features (keep only feature-specific columns from features)
base = txn_fact.alias("f") \
    .join(
        txn_features.select(feature_only_cols).alias("feat"),
        "transaction_id",
        "inner"
    )

# step 2: account + customer (inner join — silver guarantees referential integrity)
base = base.alias("t1") \
    .join(acct_cust.alias("ac"),
          col("t1.source_account_id") == col("ac.account_id"),
          "inner") \
    .drop(col("ac.account_id")) \
    .drop(col("ac.customer_id"))

# step 3: card (left join — not all txns use cards)
base = base.alias("t2") \
    .join(card_lookup.alias("cd"),
          col("t2.card_id") == col("cd.card_id"),
          "left") \
    .drop(col("cd.card_id"))

# step 4: merchant (left join — not all txns have merchants)
base = base.alias("t3") \
    .join(merchant_lookup.alias("m"),
          col("t3.merchant_id") == col("m.merchant_id"),
          "left") \
    .drop(col("m.merchant_id"))

# step 5: device (left join — not all txns have device fingerprint)
base = base.alias("t4") \
    .join(cust_device.alias("dv"),
          (col("t4.customer_id") == col("dv.customer_id")) &
          (col("t4.device_id") == col("dv.device_id")),
          "left") \
    .drop(col("dv.customer_id")) \
    .drop(col("dv.device_id"))

# step 6: calendar (inner join — every txn has a valid date)
base = base.alias("t5") \
    .join(date_lookup.alias("dt"),
          col("t5.transaction_date").cast("date") == col("dt.date_key"),
          "inner") \
    .drop(col("dt.date_key"))

row_count = base.count()
col_count = len(base.columns)
print(f"\nBase table assembled: {row_count:,} rows, {col_count} columns")
print("Ready for Cell 7.")

Building enriched transaction base...
Shared columns between fact and features: 21
Feature-only columns we keep: 31

Base table assembled: 2,200,000 rows, 97 columns
Ready for Cell 7.


In [0]:
# Databricks notebook source
# =============================================================================
# CELL 7: Fraud Helper Columns
# =============================================================================
# Silver flags (high_velocity_flag, shared_device_flag, off_hours_flag,
# new_merchant_flag, high_risk_amount_flag) are INT (1=True, 0=False).
# Silver time features (hour_of_day, day_of_week, etc.) are INT.
# Silver amount columns are DECIMAL/DOUBLE.
# =============================================================================

# --- hour bucket (uses hour_of_day from Silver, which is INT) ---
base = base.withColumn(
    "hour_bucket",
    when((col("hour_of_day") >= 6) & (col("hour_of_day") <= 11), "Morning")
    .when((col("hour_of_day") >= 12) & (col("hour_of_day") <= 16), "Afternoon")
    .when((col("hour_of_day") >= 17) & (col("hour_of_day") <= 21), "Evening")
    .otherwise("Night")
)

# --- tenure ---
base = base \
    .withColumn("cust_tenure_days",
                datediff(col("transaction_date").cast("date"), col("cust_onboarded"))) \
    .withColumn("cust_tenure_bucket",
                when(col("cust_tenure_days") < 90, "New")
                .when(col("cust_tenure_days") <= 730, "Established")
                .otherwise("Veteran")) \
    .withColumn("acct_age_days",
                datediff(col("transaction_date").cast("date"), col("account_opened")))

# --- amount vs baseline ---
base = base.withColumn(
    "amount_vs_30d_avg",
    when(col("avg_customer_amount_30d").isNotNull() & (col("avg_customer_amount_30d") > 0),
         round(col("transaction_amount") / col("avg_customer_amount_30d"), 2))
    .otherwise(None)
)

# --- merchant category risk weight ---
base = base.withColumn(
    "mcc_risk_weight",
    when(col("merchant_category") == "Gaming", 0.90)
    .when(col("merchant_category") == "Ecommerce", 0.60)
    .when(col("merchant_category") == "Telecom", 0.50)
    .when(col("merchant_category") == "Financial", 0.40)
    .when(col("merchant_category") == "Travel", 0.35)
    .when(col("merchant_category") == "Fuel", 0.25)
    .when(col("merchant_category") == "Restaurant", 0.20)
    .when(col("merchant_category") == "Retail", 0.20)
    .when(col("merchant_category") == "Clothing", 0.20)
    .when(col("merchant_category") == "Healthcare", 0.15)
    .when(col("merchant_category") == "Utilities", 0.10)
    .when(col("merchant_category") == "Grocery", 0.10)
    .otherwise(0.30)
)

# --- first txn on device ---
dev_window = Window.partitionBy("customer_id", "device_id").orderBy("transaction_date")
base = base.withColumn("is_first_txn_on_device", row_number().over(dev_window) == 1)

# --- cross risk tier ---
# Silver flags are INT: 1 = True, 0 = False. Compare to 1, not True.
base = base.withColumn(
    "cross_risk_tier",
    when((col("high_velocity_flag") == 1) & (col("shared_device_flag") == 1), "HIGH_CROSS")
    .when((col("off_hours_flag") == 1) & (col("new_merchant_flag") == 1), "MEDIUM_CROSS")
    .when(col("high_velocity_flag") == 1, "VELOCITY")
    .when(col("shared_device_flag") == 1, "DEVICE_SHARE")
    .otherwise("NONE")
)

print("Fraud helpers computed.")

print("\nHour bucket:")
base.groupBy("hour_bucket").count().orderBy("hour_bucket").show()

print("Tenure bucket:")
base.groupBy("cust_tenure_bucket").count().orderBy("cust_tenure_bucket").show()

print("Cross-risk tier:")
base.groupBy("cross_risk_tier").count().orderBy(col("count").desc()).show()

Fraud helpers computed.

Hour bucket:
+-----------+-------+
|hour_bucket|  count|
+-----------+-------+
|      Night|2200000|
+-----------+-------+

Tenure bucket:
+------------------+-------+
|cust_tenure_bucket|  count|
+------------------+-------+
|       Established| 648232|
|               New| 446167|
|           Veteran|1105601|
+------------------+-------+

Cross-risk tier:
+---------------+-------+
|cross_risk_tier|  count|
+---------------+-------+
|   DEVICE_SHARE|1461186|
|           NONE| 610411|
|   MEDIUM_CROSS| 128403|
+---------------+-------+



In [0]:
# Databricks notebook source
# =============================================================================
# CELL 8: Write gold_base_enriched_transactions
# =============================================================================

target = f"{gold}.gold_base_enriched_transactions"

# The date dimension join in Cell 6 brought in is_weekend and is_month_end
# which already exist from Silver features. We need to drop the SECOND
# occurrence of each (the date dim versions) before selecting.
# 
# The simplest approach: create a new DataFrame selecting columns by index,
# skipping the duplicate columns that appear later in the list.

all_cols = base.columns

# Find which indices have duplicate column names
seen = {}
drop_indices = []
for i, c in enumerate(all_cols):
    if c in seen:
        drop_indices.append(i)  # this is the duplicate, drop it
    else:
        seen[c] = i

# Keep only columns whose index is NOT in drop_indices
keep_cols = [all_cols[i] for i in range(len(all_cols)) if i not in drop_indices]

# Select using column objects (not strings) to avoid ambiguity
base_clean = base.select([col(base.columns[i]) for i in range(len(all_cols)) if i not in drop_indices])

# Now build Gold from the clean base
gold_df = base_clean.select(
    "transaction_id",
    "transaction_date",
    "transaction_timestamp",
    "transaction_type",
    "transaction_amount",
    "transaction_status",
    "channel",
    "authorisation_code",
    "response_code",

    "customer_id",
    "cust_full_name",
    "customer_segment",
    "cust_province",
    "cust_city",
    "cust_employment",
    "cust_income_band",
    "cust_kyc_score",
    "cust_onboarded",
    "cust_tenure_days",
    "cust_tenure_bucket",

    "source_account_id",
    "account_type",
    "account_status",
    "account_opened",
    "acct_age_days",
    "acct_balance",
    "acct_credit_limit",
    "dest_account_id",

    "card_id",
    "card_type",
    "card_network",
    "card_status",
    "card_contactless",

    "merchant_id",
    "merchant_name",
    "mcc",
    "merchant_category",
    "merch_province",
    "merch_risk_band",
    "mcc_risk_weight",
    "merch_director",
    "merch_director_id",

    "device_id",
    "device_type",
    "dev_os",
    "dev_mfr",
    "device_model",
    "dev_risk_score",
    "is_emulator",
    "is_rooted",
    "dev_link_type",
    "is_primary_device",
    "is_first_txn_on_device",

    "hour_of_day",
    "hour_bucket",
    "day_of_week",
    "week_of_year",
    "month_number",
    "is_weekend",
    "is_month_end",

    "dow",
    "cal_month",
    "cal_month_name",
    "cal_quarter",
    "cal_year",
    "is_holiday",

    "customer_txn_count_1d",
    "customer_txn_count_7d",
    "customer_txn_count_30d",
    "customer_amount_sum_1d",
    "customer_amount_sum_7d",
    "customer_amount_sum_30d",
    "avg_customer_amount_7d",
    "avg_customer_amount_30d",
    "amount_vs_customer_avg",

    "merchant_txn_count_1d",
    "merchant_txn_count_7d",
    "merchant_volume_7d",
    "merchant_customer_count",

    "device_customer_count",
    "device_account_count",

    "high_velocity_flag",
    "shared_device_flag",
    "high_risk_amount_flag",
    "new_merchant_flag",
    "off_hours_flag",

    "amount_vs_30d_avg",
    "cross_risk_tier"
)

print(f"Writing to {target}...")
print(f"Rows: {gold_df.count():,}")
print(f"Columns: {len(gold_df.columns)}")

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("transaction_date") \
    .saveAsTable(target)

print(f"Table written: {target}")

Writing to ubuntu_bank_fraud.gold.gold_base_enriched_transactions...
Rows: 2,200,000
Columns: 88
Table written: ubuntu_bank_fraud.gold.gold_base_enriched_transactions


In [0]:
# Databricks notebook source
# =============================================================================
# CELL 8: Write gold_base_enriched_transactions
# =============================================================================

target = f"{gold}.gold_base_enriched_transactions"

# After Cell 6 joins, we have duplicate columns from the date dimension.
# Silver already provides: is_weekend, is_month_end
# Date dimension also brought: is_weekend, is_month_end, is_holiday, dow, etc.
# We keep the Silver versions and only take is_holiday from the date dim.
# The cleanest fix: drop the date dim duplicates by renaming them first in Cell 5.
# But since Cell 5 already ran, we fix it here by selecting only what we need.

gold_df = base.select(
    "transaction_id",
    "transaction_date",
    "transaction_timestamp",
    "transaction_type",
    "transaction_amount",
    "transaction_status",
    "channel",
    "authorisation_code",
    "response_code",

    "customer_id",
    "cust_full_name",
    "customer_segment",
    "cust_province",
    "cust_city",
    "cust_employment",
    "cust_income_band",
    "cust_kyc_score",
    "cust_onboarded",
    "cust_tenure_days",
    "cust_tenure_bucket",

    "source_account_id",
    "account_type",
    "account_status",
    "account_opened",
    "acct_age_days",
    "acct_balance",
    "acct_credit_limit",
    "dest_account_id",

    "card_id",
    "card_type",
    "card_network",
    "card_status",
    "card_contactless",

    "merchant_id",
    "merchant_name",
    "mcc",
    "merchant_category",
    "merch_province",
    "merch_risk_band",
    "mcc_risk_weight",
    "merch_director",
    "merch_director_id",

    "device_id",
    "device_type",
    "dev_os",
    "dev_mfr",
    "device_model",
    "dev_risk_score",
    "is_emulator",
    "is_rooted",
    "dev_link_type",
    "is_primary_device",
    "is_first_txn_on_device",

    "hour_of_day",
    "hour_bucket",
    "day_of_week",
    "week_of_year",
    "month_number",
    # is_weekend and is_month_end come from Silver features only
    col("t5.is_weekend").alias("is_weekend"),
    col("t5.is_month_end").alias("is_month_end"),

    # date dimension columns (unique to date dim, no conflict)
    "dow",
    "cal_month",
    "cal_month_name",
    "cal_quarter",
    "cal_year",
    "is_holiday",

    "customer_txn_count_1d",
    "customer_txn_count_7d",
    "customer_txn_count_30d",
    "customer_amount_sum_1d",
    "customer_amount_sum_7d",
    "customer_amount_sum_30d",
    "avg_customer_amount_7d",
    "avg_customer_amount_30d",
    "amount_vs_customer_avg",

    "merchant_txn_count_1d",
    "merchant_txn_count_7d",
    "merchant_volume_7d",
    "merchant_customer_count",

    "device_customer_count",
    "device_account_count",

    "high_velocity_flag",
    "shared_device_flag",
    "high_risk_amount_flag",
    "new_merchant_flag",
    "off_hours_flag",

    "amount_vs_30d_avg",
    "cross_risk_tier"
)

print(f"Writing to {target}...")
print(f"Rows: {gold_df.count():,}")
print(f"Columns: {len(gold_df.columns)}")

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("transaction_date") \
    .saveAsTable(target)

print(f"Table written: {target}")

Writing to ubuntu_bank_fraud.gold.gold_base_enriched_transactions...
Rows: 2,200,000
Columns: 88
Table written: ubuntu_bank_fraud.gold.gold_base_enriched_transactions


In [0]:
# Databricks notebook source
# =============================================================================
# CELL 10: Validation
# =============================================================================

target = f"{gold}.gold_base_enriched_transactions"
g = spark.table(target)

print("=" * 55)
print("VALIDATION: gold_base_enriched_transactions")
print("=" * 55)

# 1. row count
gold_rows = g.count()
silver_rows = txn_fact.count()
status = "OK" if gold_rows == silver_rows else "MISMATCH"
print(f"\n[1] Row count: Gold={gold_rows:,}  Silver={silver_rows:,}  [{status}]")

# 2. primary key uniqueness
dupes = g.groupBy("transaction_id").count().filter(col("count") > 1).count()
print(f"[2] Duplicate PKs: {dupes} {'[OK]' if dupes == 0 else '[FAIL]'}")

# 3. join completeness
null_cust = g.filter(col("cust_full_name").isNull()).count()
null_acct = g.filter(col("account_type").isNull()).count()
print(f"[3] Join completeness:")
print(f"    null customers: {null_cust} {'[OK]' if null_cust == 0 else '[FAIL]'}")
print(f"    null accounts:  {null_acct} {'[OK]' if null_acct == 0 else '[FAIL]'}")

# 4. date range
dates = g.agg(min("transaction_date").alias("d1"), max("transaction_date").alias("d2")).collect()[0]
print(f"[4] Date range: {dates.d1} to {dates.d2}")

# 5. sample
print("\n[5] Sample:")
g.select("transaction_id", "hour_bucket", "cust_tenure_bucket", "amount_vs_30d_avg", "cross_risk_tier").show(5, truncate=False)

# 6. cross-risk distribution
print("[6] Cross-risk distribution:")
g.groupBy("cross_risk_tier").count().orderBy(col("count").desc()).show(truncate=False)

# final
all_ok = (gold_rows == silver_rows) and (dupes == 0) and (null_cust == 0) and (null_acct == 0)
print(f"\n{'='*55}")
print("ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED")
print(f"{'='*55}")

VALIDATION: gold_base_enriched_transactions

[1] Row count: Gold=2,200,000  Silver=2,200,000  [OK]
[2] Duplicate PKs: 0 [OK]
[3] Join completeness:
    null customers: 0 [OK]
    null accounts:  0 [OK]
[4] Date range: 2024-01-01 00:00:00 to 2025-12-31 00:00:00

[5] Sample:
+--------------+-----------+------------------+-----------------+---------------+
|transaction_id|hour_bucket|cust_tenure_bucket|amount_vs_30d_avg|cross_risk_tier|
+--------------+-----------+------------------+-----------------+---------------+
|TXN01167082   |Night      |Veteran           |0.97             |DEVICE_SHARE   |
|TXN01533983   |Night      |Veteran           |0.67             |DEVICE_SHARE   |
|TXN01196930   |Night      |Established       |2.7              |DEVICE_SHARE   |
|TXN00342831   |Night      |Veteran           |0.25             |DEVICE_SHARE   |
|TXN00915485   |Night      |Veteran           |0.59             |NONE           |
+--------------+-----------+------------------+-----------------+-----

In [0]:
# Databricks notebook source
# =============================================================================
# CELL 11: Output Contract
# =============================================================================

print("""
================================================================
OUTPUT CONTRACT: gold_base_enriched_transactions
================================================================

TABLE:     ubuntu_bank_fraud.gold.gold_base_enriched_transactions
GRAIN:     One row per transaction (2,200,000 rows)
PARTITION: transaction_date
Z-ORDER:   (customer_id, merchant_id)

DOWNSTREAM MARTS:

  gold_fiu_customer_velocity_network
  gold_customer_risk_profile_cube
  gold_merchant_syndicate_anomaly_mart
  gold_device_linkage_risk_mart

GUARANTEES:
  - transaction_id is unique
  - customer and account context never null
  - Silver features passed through unchanged
  - date range: 2024-01-01 to 2025-12-31
================================================================
""")


OUTPUT CONTRACT: gold_base_enriched_transactions

TABLE:     ubuntu_bank_fraud.gold.gold_base_enriched_transactions
GRAIN:     One row per transaction (2,200,000 rows)
PARTITION: transaction_date
Z-ORDER:   (customer_id, merchant_id)

DOWNSTREAM MARTS:

  gold_fiu_customer_velocity_network
  gold_customer_risk_profile_cube
  gold_merchant_syndicate_anomaly_mart
  gold_device_linkage_risk_mart

GUARANTEES:
  - transaction_id is unique
  - customer and account context never null
  - Silver features passed through unchanged
  - date range: 2024-01-01 to 2025-12-31

